# Visualization & Analysis

Explore SegmentAnyTree results: plot segmentation maps, analyze per-tree metrics,
and prepare data for 3D viewers.

## Visualization approaches

| Tool | Best for | Format |
|------|----------|--------|
| **matplotlib** (this notebook) | Quick 2D plots, histograms, metrics | Any |
| **[CloudCompare](https://www.danielgm.net/cc/)** | Interactive 3D exploration | .copc.laz, .las |
| **[QGIS](https://qgis.org/)** | GIS overlay, map context | .copc.laz (native) |
| **[Potree](https://potree.github.io/)** / [copc.io](https://viewer.copc.io/) | Web-based 3D streaming | .copc.laz |


In [ ]:
import laspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load a segmented point cloud
results_dir = Path('/data/output/final_results')
result_files = sorted(results_dir.glob('*.copc.laz')) or sorted(results_dir.glob('*.laz')) or sorted(results_dir.glob('*.las'))

if not result_files:
    print('No results found in', results_dir)
    print('Run inference first (see 01_quickstart.ipynb)')
else:
    las = laspy.read(str(result_files[0]))
    print(f'Loaded: {result_files[0].name}')
    print(f'Points: {len(las.points):,}')
    print(f'Dimensions: {list(las.point_format.dimension_names)}')

## Instance Segmentation Map

Bird's-eye view colored by tree instance ID. Each color is a different tree.

In [ ]:
if 'las' in dir():
    # Subsample for plotting (every 10th point)
    step = max(1, len(las.points) // 500000)
    x = np.array(las.x[::step])
    y = np.array(las.y[::step])
    instances = np.array(las.PredInstance[::step])

    fig, ax = plt.subplots(figsize=(12, 10))
    scatter = ax.scatter(x, y, c=instances, cmap='tab20', s=0.1, alpha=0.6)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title(f'Instance Segmentation — {len(np.unique(instances[instances > 0]))} trees')
    ax.set_aspect('equal')
    plt.colorbar(scatter, ax=ax, label='Tree ID', shrink=0.8)
    plt.tight_layout()
    plt.show()

## Semantic Segmentation Map

Bird's-eye view colored by semantic class: unclassified (0), non-tree (1), tree (2).

In [ ]:
if 'las' in dir():
    semantic = np.array(las.PredSemantic[::step])

    colors = {0: '#999999', 1: '#2ecc71', 2: '#e74c3c'}
    labels = {0: 'Unclassified', 1: 'Non-tree', 2: 'Tree'}

    fig, ax = plt.subplots(figsize=(12, 10))
    for cls_id in sorted(colors.keys()):
        mask = semantic == cls_id
        if mask.any():
            ax.scatter(x[mask], y[mask], c=colors[cls_id], s=0.1, alpha=0.5,
                       label=f'{labels[cls_id]} ({mask.sum():,} pts)')

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('Semantic Segmentation')
    ax.set_aspect('equal')
    ax.legend(markerscale=20, loc='upper right')
    plt.tight_layout()
    plt.show()

## Points per Tree Instance

In [ ]:
if 'las' in dir():
    all_instances = np.array(las.PredInstance)
    tree_ids, counts = np.unique(all_instances[all_instances > 0], return_counts=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(counts, bins=50, edgecolor='white', alpha=0.8)
    ax.set_xlabel('Points per tree')
    ax.set_ylabel('Number of trees')
    ax.set_title(f'Point count distribution ({len(tree_ids)} trees)')
    ax.axvline(np.median(counts), color='red', linestyle='--',
               label=f'Median: {int(np.median(counts)):,}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Per-Tree Metrics

In [ ]:
if 'las' in dir():
    all_x = np.array(las.x)
    all_y = np.array(las.y)
    all_z = np.array(las.z)
    all_inst = np.array(las.PredInstance)

    tree_ids = np.unique(all_inst)
    tree_ids = tree_ids[tree_ids > 0]

    trees = []
    for tid in tree_ids:
        mask = all_inst == tid
        zz = all_z[mask]
        xx = all_x[mask]
        yy = all_y[mask]
        trees.append({
            'tree_id': int(tid),
            'n_points': int(mask.sum()),
            'height_m': float(zz.max() - zz.min()),
            'crown_x_m': float(xx.max() - xx.min()),
            'crown_y_m': float(yy.max() - yy.min()),
            'z_max': float(zz.max()),
            'centroid_x': float(xx.mean()),
            'centroid_y': float(yy.mean()),
        })

    df = pd.DataFrame(trees)
    print(f'{len(df)} trees detected')
    print()
    print(df[['tree_id', 'n_points', 'height_m', 'crown_x_m', 'crown_y_m']].describe().round(2))
    print()
    print(df.head(10).to_string(index=False))

## Tree Height Distribution

In [ ]:
if 'df' in dir() and len(df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Height distribution
    ax = axes[0]
    ax.hist(df['height_m'], bins=30, edgecolor='white', alpha=0.8, color='forestgreen')
    ax.set_xlabel('Tree height (m)')
    ax.set_ylabel('Count')
    ax.set_title('Tree Height Distribution')
    ax.axvline(df['height_m'].median(), color='red', linestyle='--',
               label=f'Median: {df["height_m"].median():.1f} m')
    ax.legend()

    # Crown diameter (mean of x and y extent)
    ax = axes[1]
    crown_diam = (df['crown_x_m'] + df['crown_y_m']) / 2
    ax.hist(crown_diam, bins=30, edgecolor='white', alpha=0.8, color='steelblue')
    ax.set_xlabel('Crown diameter (m)')
    ax.set_ylabel('Count')
    ax.set_title('Crown Diameter Distribution')
    ax.axvline(crown_diam.median(), color='red', linestyle='--',
               label=f'Median: {crown_diam.median():.1f} m')
    ax.legend()

    plt.tight_layout()
    plt.show()

## 3D Visualization with External Tools

For interactive 3D exploration, use one of these tools with your `.copc.laz` output files:

### CloudCompare
1. Open your `.copc.laz` file
2. Select **PredInstance** as the active scalar field
3. Set color ramp to **Random** for distinct tree colors

### QGIS (3.26+)
1. Drag `.copc.laz` into the map canvas
2. Style by **PredInstance** attribute with Classification renderer
3. COPC spatial index enables efficient partial loading

### Web Viewers
- **[copc.io viewer](https://viewer.copc.io/)**: paste a public COPC URL
- **[Potree](https://potree.github.io/)**: self-hosted web viewer

See the [Scientific Workflow](../docs/workflow.md) guide for detailed visualization instructions.